In [1]:
import numpy as np
from pathlib import Path
import pandas as pd

saved_dir = Path(r"C:\Users\nb0801\Documents\GitHub\IDS-CAN-Bus-In-Vehicle-Networks-Based-on-the-Statistical-Characteristics-of-Attacks\saved_data\ROAD")

train_dfs = pd.read_pickle(saved_dir / "train_dfs.pkl")
test_dfs = pd.read_pickle(saved_dir / "test_dfs.pkl")

print("Reloaded train_dfs and test_dfs from saved_data/")
print("Train dfs:", len(train_dfs))
print("Test dfs:", len(test_dfs))
print("Train sizes:", [len(df) for df in train_dfs])
print("Test sizes:", [len(df) for df in test_dfs])

Reloaded train_dfs and test_dfs from saved_data/
Train dfs: 28
Test dfs: 28
Train sizes: [41580, 39911, 42657, 40945, 25197, 24186, 8058, 8024, 49098, 47143, 62576, 60064, 121688, 116802, 13400, 12862, 47249, 45352, 48501, 46554, 39720, 38128, 73526, 70575, 46872, 44992, 2393018, 850900]
Test sizes: [10394, 9977, 10664, 10236, 6299, 6046, 2014, 2006, 12274, 11785, 15644, 15016, 30421, 29200, 3349, 3215, 11812, 11338, 12125, 11638, 9930, 9531, 18381, 17643, 11718, 11247, 598254, 212724]


In [2]:
def prepare_windowed_dataset(
    train_dfs,
    test_dfs,
    data_col="Data",
    id_onehot_col="ID_onehot",
    attack_col="Attack",
    window_size=32,
    step=2,
):
    """
    Creates sliding-window datasets for training and testing.

    Returns
    -------
    windows : ndarray
        Training windows of shape (N, 54, 8)

    windows_test : ndarray
        Test windows of shape (M, 54, 8)

    attack_onehot : ndarray
        Training labels as one-hot vectors (R,T)

    attack_onehot_test : ndarray
        Test labels as one-hot vectors (R,T)
    """

    # ----------------------------------------------------
    # Convert CAN payload to uint8 array
    # ----------------------------------------------------
    def data_to_uint8_array(data):
        if isinstance(data, bytes):
            arr = np.frombuffer(data, dtype=np.uint8)

        elif isinstance(data, str):
            s = data.strip()

            if s.startswith(("0x", "0X")):
                s = s[2:]

            if any(ch in s for ch in (" ", "-", ":", ",")):
                parts = [
                    tok for tok in
                    s.replace("-", " ")
                     .replace(":", " ")
                     .replace(",", " ")
                     .split()
                    if tok
                ]

                if all(len(tok) == 2 for tok in parts):
                    arr = np.array([int(tok, 16) for tok in parts],
                                   dtype=np.uint8)
                else:
                    arr = np.frombuffer(
                        bytes.fromhex("".join(parts)),
                        dtype=np.uint8,
                    )

            else:
                arr = np.frombuffer(bytes.fromhex(s), dtype=np.uint8)

        elif isinstance(data, (list, tuple, np.ndarray, pd.Series)):
            arr = np.asarray(data, dtype=np.uint8).flatten()

        elif isinstance(data, int):
            arr = np.frombuffer(
                data.to_bytes(8, byteorder="big", signed=False),
                dtype=np.uint8,
            )

        else:
            raise TypeError(f"Unsupported Data type: {type(data)}")

        if arr.size < 8:
            arr = np.pad(arr, (0, 8 - arr.size))
        elif arr.size > 8:
            arr = arr[:8]

        return arr

    # ----------------------------------------------------
    # Process one dataset (train or test)
    # ----------------------------------------------------
    def process(dfs):

        payload_windows = []
        id_windows = []
        labels = []

        for df in dfs:

            payload = np.stack(
                [data_to_uint8_array(x) for x in df[data_col]]
            )

            ids = np.stack(df[id_onehot_col].tolist())

            if ids.shape[1:] != (22, 8):
                raise ValueError(
                    f"{id_onehot_col} must contain arrays of shape (22,8)"
                )

            attacks = df[attack_col].to_numpy()

            for start in range(0, len(df) - window_size, step):

                end = start + window_size

                # 32×8 payload window
                payload_window = payload[start:end]

                # summed 22×8 ID matrix
                id_sum = ids[start:end].sum(axis=0, dtype=np.uint16)

                # combine -> 54×8
                combined = np.concatenate(
                    [payload_window.astype(np.uint16), id_sum],
                    axis=0,
                )

                payload_windows.append(combined)

                # label window
                labels.append(
                    'T' if np.any(attacks[start:end] == 'T') else 'R'
                )

        payload_windows = np.asarray(payload_windows, dtype=np.uint16)
        labels = np.asarray(labels)

        # one-hot encoding
        attack_onehot = np.zeros((len(labels), 2), dtype=np.uint8)
        attack_onehot[labels == "R", 0] = 1
        attack_onehot[labels == "T", 1] = 1

        return payload_windows, attack_onehot

    # ----------------------------------------------------
    # Build datasets
    # ----------------------------------------------------
    windows, attack_onehot = process(train_dfs)
    windows_test, attack_onehot_test = process(test_dfs)

    return (
        windows,
        windows_test,
        attack_onehot,
        attack_onehot_test,
    )

windows, windows_test, attack_onehot, attack_onehot_test = prepare_windowed_dataset(
    train_dfs,
    test_dfs,
    window_size=32,
    step=2,
)

print(windows.shape)
print(windows_test.shape)
print(attack_onehot.shape)
print(attack_onehot_test.shape)

(2229345, 54, 8)
(556998, 54, 8)
(2229345, 2)
(556998, 2)


In [15]:
import torch
from torch.utils.data import Dataset


class CANDataset(Dataset):
    def __init__(
        self,
        windows,
        attack_onehot,
        train=True,
        is_spiking=False,
        time_window=100,
    ):
        # Convert to tensors
        self.windows = torch.as_tensor(windows, dtype=torch.float32)

        # Add channel dimension:
        # (N, H, W) -> (N, 1, H, W)
        if self.windows.ndim == 3:
            self.windows = self.windows.unsqueeze(1)

        attack_onehot = np.asarray(attack_onehot)

        # Convert one-hot -> class index
        if attack_onehot.ndim == 2:
            attack_labels = np.argmax(attack_onehot, axis=1)
        else:
            attack_labels = attack_onehot

        self.labels = torch.tensor(attack_labels, dtype=torch.long)

        self.is_spiking = is_spiking
        self.time_window = time_window

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, index):

        img = self.windows[index]
        label = self.labels[index]

        if self.is_spiking:
            # Optional normalization if inputs are not already in [0,1]
            #img = (img - img.min()) / (img.max() - img.min() + 1e-8)

            # Generate spike train
            img = (
                torch.rand(self.time_window, *img.shape) < img
            ).float()

        return img, label

from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

windows = scaler.fit_transform(
    windows.reshape(-1, windows.shape[-1])
).reshape(windows.shape)

windows_test = scaler.transform(
    windows_test.reshape(-1, windows_test.shape[-1])
).reshape(windows_test.shape)

from torch.utils.data import DataLoader

canbus_train = CANDataset(windows, attack_onehot, train=True, is_spiking=False)
train_loader = DataLoader(canbus_train, batch_size=2000, shuffle=True)

canbus_test = CANDataset(windows_test, attack_onehot_test, train=False, is_spiking=False)
test_loader = DataLoader(canbus_test, batch_size=2000, shuffle=False)

In [16]:
import torch.nn as nn
from tqdm.auto import tqdm

def create_ann(
    input_shape=(54, 8),
    num_classes=2,
    conv1_filters=32,
    conv2_filters=64,
    conv1_kernel_size=(3, 3),
    conv2_kernel_size=(3, 3),
):

    h, w = input_shape

    model = nn.Sequential(
        nn.Conv2d(
            1,
            conv1_filters,
            kernel_size=conv1_kernel_size,
        ),
        nn.ReLU(),

        nn.MaxPool2d((2,2)),

        nn.Conv2d(
            conv1_filters,
            conv2_filters,
            kernel_size=conv2_kernel_size,
        ),
        nn.ReLU(),

        nn.Flatten(),
    )

    # Automatically determine flatten size
    with torch.no_grad():
        dummy = torch.zeros(1, 1, h, w)
        flatten_size = model(dummy).shape[1]

    model.append(nn.Linear(flatten_size, 64))
    model.append(nn.ReLU())
    model.append(nn.Linear(64, num_classes))

    return model

import torch
import torch.nn as nn
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
import numpy as np


def train_ann(
    train_loader,
    test_loader,
    epochs=2,
    lr=1e-3,
    conv1_filters=32,
    conv2_filters=64,
    conv1_kernel_size=(3,3),
    conv2_kernel_size=(3,3),
):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = create_ann(
        conv1_filters=conv1_filters,
        conv2_filters=conv2_filters,
        conv1_kernel_size=conv1_kernel_size,
        conv2_kernel_size=conv2_kernel_size,
    ).to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # ----------------------------
    # Training
    # ----------------------------

    model.train()

    
    for epoch in range(epochs):

        model.train()

        running_loss = 0
        correct = 0
        total = 0

        pbar = tqdm(
            train_loader,
            desc=f"Epoch {epoch+1}/{epochs}",
            leave=False,
        )

        for x, y in pbar:

            x = x.to(device)
            y = y.to(device)

            optimizer.zero_grad()

            output = model(x)

            loss = criterion(output, y)

            loss.backward()

            optimizer.step()

            running_loss += loss.item()

            pred = output.argmax(dim=1)

            correct += (pred == y).sum().item()
            total += y.size(0)

            accuracy = correct / total

            pbar.set_postfix(
                loss=f"{running_loss/(pbar.n+1):.4f}",
                acc=f"{accuracy:.4f}",
            )

        print(
            f"Epoch {epoch+1}/{epochs}"
            f"  Loss={running_loss/len(train_loader):.4f}"
            f"  Acc={accuracy:.4f}"
        )

    # ----------------------------
    # Evaluation
    # ----------------------------

    model.eval()

    y_true = []
    y_pred = []

    with torch.no_grad():

        for x, y in test_loader:

            x = x.to(device)

            output = model(x)

            pred = torch.argmax(output, dim=1)

            y_true.extend(y.numpy())
            y_pred.extend(pred.cpu().numpy())

    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="weighted")
    recall = recall_score(y_true, y_pred, average="weighted")
    f1 = f1_score(y_true, y_pred, average="weighted")

    report = classification_report(
        y_true,
        y_pred,
        target_names=["R", "T"],
        output_dict=True,
        zero_division=0,
    )

    cm = confusion_matrix(y_true, y_pred)

    fp = cm.sum(axis=0) - np.diag(cm)
    fn = cm.sum(axis=1) - np.diag(cm)
    tp = np.diag(cm)
    tn = cm.sum() - (fp + fn + tp)

    fpr = fp / (fp + tn)
    fnr = fn / (fn + tp)

    metrics = {
        "accuracy": round(accuracy, 4),
        "weighted_precision": round(precision, 4),
        "weighted_recall": round(recall, 4),
        "weighted_f1": round(f1, 4),
        "classification_report": report,
        "confusion_matrix": cm,
        "false_positive_rate": {
            "R": round(float(fpr[0]), 4),
            "T": round(float(fpr[1]), 4),
        },
        "false_negative_rate": {
            "R": round(float(fnr[0]), 4),
            "T": round(float(fnr[1]), 4),
        },
    }

    return model, metrics

In [17]:
import os
import gc
import torch
import pandas as pd

RESULTS_FILE = "cnn_hyperparameter_search.csv"

# --------------------------------------
# Load previous results if they exist
# --------------------------------------

if os.path.exists(RESULTS_FILE):
    results_df = pd.read_csv(RESULTS_FILE)

    results = results_df.to_dict("records")

    completed = set(
        zip(
            results_df.conv1_filters,
            results_df.conv2_filters,
            results_df.conv1_kernel,
            results_df.conv2_kernel,
        )
    )

    print(f"Loaded {len(completed)} completed architectures.")

else:
    results = []
    completed = set()

In [19]:
import gc
import pandas as pd

conv1_filters_list = [16, 32, 64]
conv2_filters_list = [16, 32, 64]

kernel_sizes = [
    (2,2),
    (3,3),
    (4,4),
]

results = []

for conv1_filters in conv1_filters_list:
    for conv2_filters in conv2_filters_list:
        for conv1_kernel in kernel_sizes:
            for conv2_kernel in kernel_sizes:

                architecture = (
                    conv1_filters,
                    conv2_filters,
                    str(conv1_kernel),
                    str(conv2_kernel),
                )

                if architecture in completed:
                    print(f"Skipping {architecture}")
                    continue

                print(f"Testing {architecture}")

                try:

                    model, metrics = train_ann(
                        train_loader,
                        test_loader,
                        epochs=2,
                        conv1_filters=conv1_filters,
                        conv2_filters=conv2_filters,
                        conv1_kernel_size=conv1_kernel,
                        conv2_kernel_size=conv2_kernel,
                    )

                    result = {
                        "conv1_filters": conv1_filters,
                        "conv2_filters": conv2_filters,
                        "conv1_kernel": str(conv1_kernel),
                        "conv2_kernel": str(conv2_kernel),
                        "accuracy": metrics["accuracy"],
                        "precision": metrics["weighted_precision"],
                        "recall": metrics["weighted_recall"],
                        "f1": metrics["weighted_f1"],
                        "status": "Success",
                        "error": "",
                    }

                except Exception as e:

                    result = {
                        "conv1_filters": conv1_filters,
                        "conv2_filters": conv2_filters,
                        "conv1_kernel": str(conv1_kernel),
                        "conv2_kernel": str(conv2_kernel),
                        "accuracy": None,
                        "precision": None,
                        "recall": None,
                        "f1": None,
                        "status": "Failed",
                        "error": str(e),
                    }

                results.append(result)

                # Save immediately after every architecture
                pd.DataFrame(results).to_csv(RESULTS_FILE, index=False)

                completed.add(architecture)

                try:
                    del model
                except:
                    pass

                gc.collect()

                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

results_df = pd.DataFrame(results)
results_df = results_df.sort_values(
    "f1",
    ascending=False,
    na_position="last",
)

print(results_df)

Testing (16, 16, '(2, 2)', '(2, 2)')


Epoch 1/2:   0%|          | 0/1115 [00:00<?, ?it/s]

Epoch 1/2  Loss=0.3957  Acc=0.8224


Epoch 2/2:   0%|          | 0/1115 [00:00<?, ?it/s]

Epoch 2/2  Loss=0.2482  Acc=0.8963
Testing (16, 16, '(2, 2)', '(3, 3)')


...

Epoch 1/2  Loss=0.3799  Acc=0.8312


Epoch 2/2:   0%|          | 0/1115 [00:00<?, ?it/s]

Epoch 2/2  Loss=0.2139  Acc=0.9128
Testing (16, 16, '(2, 2)', '(4, 4)')
Testing (16, 16, '(3, 3)', '(2, 2)')


Epoch 1/2:   0%|          | 0/1115 [00:00<?, ?it/s]

KeyboardInterrupt: 